# SmartBite Date-Synth 1000 Colab Workflow

This notebook runs the same date-recognition fine-tuning flow in Google Colab:
1. Mount Drive
2. Unzip your `minidatesynth1000.zip` dataset
3. Validate PP-OCR dataset layout (`train_images/`, `val_images/`, labels)
4. Install Paddle + PaddleOCR training deps
5. Run training with PP-OCRv5 config


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [14]:
!rm -rf /content/dataset
!mkdir -p /content/dataset

In [ ]:
!unzip -q "/content/drive/My Drive/sb-colab/ppocrv5_date_synth_dataset.zip" -d "/content/dataset/"

In [17]:
from pathlib import Path
DATASET_ROOT = Path("/content/dataset/ppocrv5_date_synth_dataset")  # adjust exact folder


In [18]:
print("DATASET_ROOT =", DATASET_ROOT)
print("train_label lines =", sum(1 for _ in open(DATASET_ROOT / "train_label.txt", "r", encoding="utf-8")))
print("val_label lines =", sum(1 for _ in open(DATASET_ROOT / "val_label.txt", "r", encoding="utf-8")))


DATASET_ROOT = /content/dataset/ppocrv5_date_synth_dataset
train_label lines = 115200
val_label lines = 12800


In [19]:
from pathlib import Path

def find_dataset_root(base: Path) -> Path:
    direct = base
    if (direct / 'train_label.txt').exists() and (direct / 'val_label.txt').exists():
        return direct

    for candidate in sorted(base.rglob('*')):
        if not candidate.is_dir():
            continue
        if (candidate / 'train_label.txt').exists() and (candidate / 'val_label.txt').exists():
            return candidate

    raise FileNotFoundError('Could not find dataset root containing train_label.txt and val_label.txt')

BASE_UNZIP_DIR = Path('/content/dataset')
DATASET_ROOT = find_dataset_root(BASE_UNZIP_DIR)
print('DATASET_ROOT =', DATASET_ROOT)

for split in ('train', 'val'):
    label_file = DATASET_ROOT / f'{split}_label.txt'
    lines = [line for line in label_file.read_text(encoding='utf-8').splitlines() if line.strip()]
    missing = 0
    for line in lines:
        rel_path = line.split('\t', 1)[0]
        if not (DATASET_ROOT / rel_path).exists():
            missing += 1
    print(f'{split}: labels={len(lines)} missing_files={missing}')


DATASET_ROOT = /content/dataset/ppocrv5_date_synth_dataset
train: labels=115200 missing_files=115200
val: labels=12800 missing_files=12800


## Wheelhouse (No Internet Installs in Colab)

This notebook installs dependencies from Drive-hosted wheels only.

Expected folders in Drive:
- `/content/drive/My Drive/wheels/colab-cu118` for GPU runtime
- `/content/drive/My Drive/wheels/colab-cpu` for CPU runtime

Build these wheel folders once on your machine, upload to Drive, then run this notebook.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Use existing pretrained model from Drive (no Colab-side download).
PRETRAINED_MODEL = Path('/content/drive/My Drive/sb-colab/models/PP-OCRv5_server_rec_pretrained.pdparams')
assert PRETRAINED_MODEL.exists(), f'Missing pretrained model: {PRETRAINED_MODEL}'
print('Found pretrained model:', PRETRAINED_MODEL)

# Use PaddleOCR repo archive from Drive instead of git clone.
PADDELOCR_ZIP = Path('/content/drive/My Drive/sb-colab/PaddleOCR.zip')
PADDELOCR_DIR = Path('/content/PaddleOCR')
if not PADDELOCR_DIR.exists():
    assert PADDELOCR_ZIP.exists(), f'Missing PaddleOCR zip: {PADDELOCR_ZIP}'
    !unzip -q "/content/drive/My Drive/PaddleOCR.zip" -d /content

%cd /content/PaddleOCR

# Avoid slow model-source connectivity checks during runtime.
os.environ['PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK'] = 'True'

has_gpu = os.system('nvidia-smi > /dev/null 2>&1') == 0
print('GPU available:', has_gpu)

wheel_zip = Path('/content/drive/My Drive/colab-cu118.zip' if has_gpu else '/content/drive/My Drive/colab-cpu.zip')
wheelhouse = Path('/content/wheels/colab-cu118' if has_gpu else '/content/wheels/colab-cpu')
if not wheelhouse.exists():
    assert wheel_zip.exists(), f'Missing wheelhouse zip: {wheel_zip}'
    wheelhouse.parent.mkdir(parents=True, exist_ok=True)
    !unzip -q "{wheel_zip}" -d /content/wheels

assert wheelhouse.exists(), f'Wheelhouse did not extract correctly: {wheelhouse}'
print('Using wheelhouse:', wheelhouse)

def pip_install(args):
    cmd = [sys.executable, '-m', 'pip', 'install', '--no-index', '--find-links', str(wheelhouse), *args]
    print('>>', ' '.join(cmd))
    subprocess.run(cmd, check=True)

if has_gpu:
    pip_install(['paddlepaddle-gpu'])
else:
    pip_install(['paddlepaddle'])

pip_install(['-r', '/content/PaddleOCR/requirements.txt'])

import paddle
print('paddle version:', paddle.__version__)
print('compiled_with_cuda:', paddle.is_compiled_with_cuda())


Found pretrained model: /content/drive/My Drive/models/PP-OCRv5_server_rec_pretrained.pdparams
/content/PaddleOCR
GPU available: True
Using wheelhouse: /content/wheels/colab-cu118
>> /usr/bin/python3 -m pip install --no-index --find-links /content/wheels/colab-cu118 paddlepaddle-gpu
>> /usr/bin/python3 -m pip install --no-index --find-links /content/wheels/colab-cu118 -r /content/PaddleOCR/requirements.txt


/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


paddle version: 3.3.1
compiled_with_cuda: True


In [8]:
!python3 --version

Python 3.12.13


In [ ]:
from pathlib import Path

# Update this if your pretrained file is in a different Drive location.
PRETRAINED_MODEL = Path('/content/drive/My Drive/sb-colab/models/PP-OCRv5_server_rec_pretrained.pdparams')
BASE_CONFIG = Path('/content/PaddleOCR/configs/rec/PP-OCRv5/PP-OCRv5_server_rec.yml')
OUTPUT_DIR = Path('/content/output/ppocrv5_date_synth_1000_run')

EPOCHS = 1
BATCH_SIZE = 32
LEARNING_RATE = 0.0005

assert BASE_CONFIG.exists(), f'Missing config: {BASE_CONFIG}'
assert PRETRAINED_MODEL.exists(), f'Missing pretrained model: {PRETRAINED_MODEL}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Using config:', BASE_CONFIG)
print('Using pretrained model:', PRETRAINED_MODEL)
print('Output dir:', OUTPUT_DIR)


Using config: /content/PaddleOCR/configs/rec/PP-OCRv5/PP-OCRv5_server_rec.yml
Using pretrained model: /content/drive/My Drive/models/PP-OCRv5_server_rec_pretrained.pdparams
Output dir: /content/output/ppocrv5_date_synth_1000_run


In [10]:
import os
import shlex
import subprocess
import sys
from pathlib import Path

cmd = [
    sys.executable,
    '/content/PaddleOCR/tools/train.py',
    '-c', str(BASE_CONFIG),
    '-o',
    f'Global.pretrained_model={PRETRAINED_MODEL}',
    f'Global.save_model_dir={OUTPUT_DIR}',
    f'Global.epoch_num={EPOCHS}',
    'Global.use_gpu=True',
    'Global.print_batch_step=1',
    'Global.eval_batch_step=[0,50]',
    'Global.save_epoch_step=1',
    f'Optimizer.lr.learning_rate={LEARNING_RATE}',
    f'Train.loader.batch_size_per_card={BATCH_SIZE}',
    f'Eval.loader.batch_size_per_card={BATCH_SIZE}',
    'Train.sampler.first_bs=32',
    'Train.sampler.fix_bs=True',
    f'Train.dataset.data_dir={DATASET_ROOT}',
    f"Train.dataset.label_file_list=['{DATASET_ROOT / 'train_label.txt'}']",
    f'Eval.dataset.data_dir={DATASET_ROOT}',
    f"Eval.dataset.label_file_list=['{DATASET_ROOT / 'val_label.txt'}']",
]

if os.system('nvidia-smi > /dev/null 2>&1') != 0:
    cmd = [c if c != 'Global.use_gpu=True' else 'Global.use_gpu=False' for c in cmd]

# Auto-resume from latest checkpoint if it exists.
latest_base = Path(OUTPUT_DIR) / 'latest'
latest_params = Path(str(latest_base) + '.pdparams')
if latest_params.exists():
    cmd.append(f'Global.checkpoints={latest_base}')
    print('Resuming from:', latest_base)

print(shlex.join(cmd))
env = dict(os.environ)
env['PYTHONPATH'] = '/content/PaddleOCR'
env['PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK'] = 'True'
subprocess.run(cmd, check=True, env=env)

print('Training finished.')
print('Artifacts in:', OUTPUT_DIR)


/usr/bin/python3 /content/PaddleOCR/tools/train.py -c /content/PaddleOCR/configs/rec/PP-OCRv5/PP-OCRv5_server_rec.yml -o 'Global.pretrained_model=/content/drive/My Drive/models/PP-OCRv5_server_rec_pretrained.pdparams' Global.save_model_dir=/content/output/ppocrv5_date_synth_1000_run Global.epoch_num=1 Global.use_gpu=True Global.print_batch_step=1 Optimizer.lr.learning_rate=0.0005 Train.loader.batch_size_per_card=32 Eval.loader.batch_size_per_card=32 Train.sampler.first_bs=32 Train.sampler.fix_bs=True Train.dataset.data_dir=/content/dataset/ppocrv5_date_synth_mini1000_full 'Train.dataset.label_file_list=['"'"'/content/dataset/ppocrv5_date_synth_mini1000_full/train_label.txt'"'"']' Eval.dataset.data_dir=/content/dataset/ppocrv5_date_synth_mini1000_full 'Eval.dataset.label_file_list=['"'"'/content/dataset/ppocrv5_date_synth_mini1000_full/val_label.txt'"'"']'


CompletedProcess(args=['/usr/bin/python3', '/content/PaddleOCR/tools/train.py', '-c', '/content/PaddleOCR/configs/rec/PP-OCRv5/PP-OCRv5_server_rec.yml', '-o', 'Global.pretrained_model=/content/drive/My Drive/models/PP-OCRv5_server_rec_pretrained.pdparams', 'Global.save_model_dir=/content/output/ppocrv5_date_synth_1000_run', 'Global.epoch_num=1', 'Global.use_gpu=True', 'Global.print_batch_step=1', 'Optimizer.lr.learning_rate=0.0005', 'Train.loader.batch_size_per_card=32', 'Eval.loader.batch_size_per_card=32', 'Train.sampler.first_bs=32', 'Train.sampler.fix_bs=True', 'Train.dataset.data_dir=/content/dataset/ppocrv5_date_synth_mini1000_full', "Train.dataset.label_file_list=['/content/dataset/ppocrv5_date_synth_mini1000_full/train_label.txt']", 'Eval.dataset.data_dir=/content/dataset/ppocrv5_date_synth_mini1000_full', "Eval.dataset.label_file_list=['/content/dataset/ppocrv5_date_synth_mini1000_full/val_label.txt']"], returncode=0)

In [11]:
from pathlib import Path

print("OUTPUT_DIR:", OUTPUT_DIR)
print("exists:", OUTPUT_DIR.exists())
print("files:", [p.name for p in OUTPUT_DIR.glob("*")][:20])

log_file = OUTPUT_DIR / "train.log"
print("train.log exists:", log_file.exists())
if log_file.exists():
    print(log_file.read_text(encoding="utf-8")[-4000:])


OUTPUT_DIR: /content/output/ppocrv5_date_synth_1000_run
exists: True
files: ['iter_epoch_1.pdparams', 'latest.pdparams', 'iter_epoch_1.pdopt', 'latest.pdopt', 'iter_epoch_1.states', 'latest.states', 'train.log', 'config.yml']
train.log exists: True
NFO: epoch: [1/1], global_step: 20, lr: 0.000158, acc: 0.343750, norm_edit_dis: 0.825257, CTCLoss: 8.912472, NRTRLoss: 1.592921, loss: 10.498976, avg_reader_cost: 0.00393 s, avg_batch_cost: 0.69503 s, avg_samples: 32.0, ips: 46.04118 samples/s, eta: 0:00:11, max_mem_reserved: 7860 MB, max_mem_allocated: 5952 MB
[2026/03/26 08:51:05] ppocr INFO: epoch: [1/1], global_step: 21, lr: 0.000175, acc: 0.375000, norm_edit_dis: 0.849290, CTCLoss: 6.737882, NRTRLoss: 1.547129, loss: 8.227348, avg_reader_cost: 0.00361 s, avg_batch_cost: 0.57336 s, avg_samples: 32.0, ips: 55.81097 samples/s, eta: 0:00:10, max_mem_reserved: 7860 MB, max_mem_allocated: 5952 MB
[2026/03/26 08:51:06] ppocr INFO: epoch: [1/1], global_step: 22, lr: 0.000192, acc: 0.500000, nor

In [ ]:
from pathlib import Path
import shutil

# Uses your existing variables exactly as you set them
# OUTPUT_DIR must already point to your training output folder.
# FINAL_MODEL_DRIVE_DIR should be set by you to your desired Drive destination.
# Example (only if you haven't set it yet):
FINAL_MODEL_DRIVE_DIR = Path("/content/drive/My Drive/sb-colab")

assert OUTPUT_DIR.exists(), f"Missing OUTPUT_DIR: {OUTPUT_DIR}"
assert FINAL_MODEL_DRIVE_DIR.parent.exists(), f"Missing parent dir: {FINAL_MODEL_DRIVE_DIR.parent}"

FINAL_MODEL_DRIVE_DIR.mkdir(parents=True, exist_ok=True)

# Copy full run artifacts (recommended for resume + audit)
for item in OUTPUT_DIR.iterdir():
    target = FINAL_MODEL_DRIVE_DIR / item.name
    if item.is_dir():
        if target.exists():
            shutil.rmtree(target)
        shutil.copytree(item, target)
    else:
        shutil.copy2(item, target)

print("Saved model artifacts to:", FINAL_MODEL_DRIVE_DIR)


## Optional: Full run

For full training, set:
- `EPOCHS = 30`
- keep `BATCH_SIZE = 32` (or lower if OOM)

Then rerun the last two cells.
